In [ ]:
from llm_synthesis.transformers.pdf_extraction.mistral_pdf_extractor import (
    MistralPDFExtractor,
)

pdf_path = "<path_to_data>/data/pdf_papers/test.pdf"

pdf_bytes = open(pdf_path, "rb").read()

extracted_text = MistralPDFExtractor().forward(pdf_bytes)

print(extracted_text)

In [ ]:
with open(
    "<path_to_data>/data/txt_papers/test.txt",
    "w+",
) as f:
    f.write(extracted_text)

In [ ]:
with open("<path_to_data>/data/txt_papers/test.txt") as f:
    extracted_text = f.read()

In [ ]:
from llm_synthesis.transformers.figure_extraction.regex_figure_extractor import (
    FigureExtractorMarkdown,
)

figure_extractor = FigureExtractorMarkdown()

figures = figure_extractor.forward(extracted_text)

In [ ]:
import base64

from IPython.display import Image

# Convert base64 string to image and display
Image(base64.b64decode(figures[6].base64_data))

In [ ]:
from llm_synthesis.models.figure import FigureInfoWithPaper

figure = FigureInfoWithPaper(
    **figures[6].__dict__,
    paper_text=extracted_text,
    si_text="",
)

In [ ]:
from llm_synthesis.transformers.plot_extraction.plot_information_extraction_dspy import (
    PlotInformationExtractor,
    make_dspy_plot_information_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name

signature = make_dspy_plot_information_extractor_signature()  # Use defaults

lm = get_llm_from_name("gpt-4o-mini")

plot_information_extractor = PlotInformationExtractor(signature, lm)

plot_info = plot_information_extractor.forward(figure)

print(plot_info)
print("Plot is extractable:", plot_info.is_extractable_plot)

In [ ]:
from llm_synthesis.transformers.plot_extraction.plot_data_extraction_dspy import (
    PlotDataExtractor,
    make_dspy_plot_data_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name

signature = make_dspy_plot_data_extractor_signature()

lm = get_llm_from_name("gpt-4o-mini")

plot_information_extractor = PlotDataExtractor(signature, lm)

extracted_data = plot_information_extractor.forward((figure, ""))

print(extracted_data)

In [ ]:
from llm_synthesis.transformers.plot_extraction.plot_analysis_extraction_dspy import (
    PlotAnalysisExtractor,
    make_dspy_plot_analysis_extractor_signature,
)

signature = make_dspy_plot_analysis_extractor_signature()

lm = get_llm_from_name("gpt-4o-mini")

plot_analysis_extractor = PlotAnalysisExtractor(signature, lm)

plot_analysis = plot_analysis_extractor.forward((figure, extracted_data))

print(plot_analysis)

In [ ]:
import base64
import json
import os

from llm_synthesis.models.figure import FigureInfo


def convert_raw_image(img_path: str) -> FigureInfo:
    """
    Load an image from the given path convert it to base64-encoded string,
    then wrap it in a FigureInfo object.

    Parameters:
        path (str): Path to the image file (.png, .jpeg, .jpg)
    Returns:
        FigureInfo: An object containing the base64-encoded image data and other metadata.
    """
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in [".png", ".jpeg", ".jpg"]:
        raise ValueError(f"Unsupported image format: {ext}")

    with open(img_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    figure_info = FigureInfo(
        base64_data=encoded,
        alt_text="",
        position=0,
        context_before="",
        context_after="",
        figure_reference="",
        figure_class="",
    )

    return figure_info


img_path = "<path_of_your_image.png>"
figure_info = convert_raw_image(img_path)

In [ ]:
from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
    ClaudeLinePlotDataExtractor,
)

# Initialize the extractor
extractor = ClaudeLinePlotDataExtractor(model_name="claude-sonnet-4-20250514")

# Perform the extraction
extracted_data = extractor.forward(figure_info)
print("Extracted data:")
print(extracted_data)

# Track the cost of this operation
cost_info = extractor.get_cost()
cost_info

In [ ]:
# Orginal figure
from IPython.display import Image

Image(base64.b64decode(figure_info.base64_data))

In [ ]:
# visualize ground truth and extracted data
from llm_synthesis.transformers.plot_extraction.claude_extraction.plot_data_extraction import (
    ExtractedLinePlotData,
)
from llm_synthesis.utils.visualization import visualize_line_chart

# load ground truth coordinates
ground_truth_path = "<path_to_ground_truth.json>"  # make sure the name of each series is consistent with the extracted data

with open(ground_truth_path) as f:
    gt_coordinates = json.load(f)
    gt_extracted_data = ExtractedLinePlotData.model_validate(gt_coordinates)

# sort the names for consistent visualization
gt_extracted_data.name_to_coordinates = dict(
    sorted(
        gt_extracted_data.name_to_coordinates.items(), key=lambda item: item[0]
    )
)
extracted_data.name_to_coordinates = dict(
    sorted(extracted_data.name_to_coordinates.items(), key=lambda item: item[0])
)

visualize_line_chart(gt_extracted_data)
visualize_line_chart(extracted_data)

In [ ]:
from llm_synthesis.metrics.extraction_metric.figure_extraction_metric import (
    FigureExtractionMetric,
)

figure_extraction_metric = FigureExtractionMetric()

rmse = figure_extraction_metric(
    extracted_data, gt_extracted_data, error_metric="rmse"
)
mae = figure_extraction_metric(
    extracted_data, gt_extracted_data, error_metric="mae"
)
rmse, mae